In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=0.95)

STUDENT_NAME = "Коваль"   # <- за потреби змініть: ПІБ у шапці звіту
GROUP_NAME = ""           # <- група
SEED = 42
OUT = Path(".")
FIG = OUT / "_fig_lab3"; FIG.mkdir(exist_ok=True)

## Допоміжні функції для Word-звіту

In [ ]:
import subprocess, sys
try:
    from docx import Document
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "python-docx"])
    from docx import Document
from docx.shared import Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

FONT = "Times New Roman"

def _set_font(style, size, bold=None):
    style.font.name = FONT
    style.font.size = Pt(size)
    style.font.color.rgb = RGBColor(0, 0, 0)
    if bold is not None:
        style.font.bold = bold
    rpr = style.element.get_or_add_rPr()
    rfonts = rpr.find(qn("w:rFonts"))
    if rfonts is None:
        rfonts = OxmlElement("w:rFonts"); rpr.append(rfonts)
    for a in ("w:asciiTheme", "w:hAnsiTheme", "w:eastAsiaTheme", "w:cstheme"):
        if rfonts.get(qn(a)) is not None:
            del rfonts.attrib[qn(a)]
    for a in ("w:ascii", "w:hAnsi", "w:eastAsia", "w:cs"):
        rfonts.set(qn(a), FONT)

def new_doc(lab_no, title, variant, dataset, student="", group="", discipline="Інтелектуальний аналіз даних", info=None):
    d = Document()
    s = d.sections[0]
    s.page_width, s.page_height = Cm(21), Cm(29.7)
    s.left_margin, s.right_margin = Cm(2.5), Cm(1.5)
    s.top_margin = s.bottom_margin = Cm(2)
    _set_font(d.styles["Normal"], 12)
    d.styles["Normal"].paragraph_format.space_after = Pt(4)
    d.styles["Normal"].paragraph_format.line_spacing = 1.15
    _set_font(d.styles["Heading 1"], 13, True)
    _set_font(d.styles["Heading 2"], 12, True)
    for h in ("Heading 1", "Heading 2"):
        d.styles[h].paragraph_format.space_before = Pt(10)
        d.styles[h].paragraph_format.space_after = Pt(4)
    p = d.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run(f"ЛАБОРАТОРНА РОБОТА № {lab_no}"); r.bold = True; r.font.size = Pt(14)
    p = d.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run(title); r.bold = True
    p = d.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.add_run(f"з дисципліни «{discipline}»")
    info = info if info is not None else f"Варіант {variant} · набір даних: {dataset}"
    if student or group:
        info += "\nВиконав(ла): " + " ".join(x for x in [student, f"({group})" if group else ""] if x)
    p = d.add_paragraph(info); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    return d

def H(d, text, level=1):
    return d.add_heading(text, level=level)

def P(d, text, bold_prefix=None, italic=False):
    p = d.add_paragraph()
    if bold_prefix:
        p.add_run(bold_prefix).bold = True
    r = p.add_run(text); r.italic = italic
    p.paragraph_format.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    return p

def bullets(d, items):
    for t in items:
        p = d.add_paragraph(style="List Bullet")
        p.add_run(t)
        p.paragraph_format.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY

def numbered(d, items):
    for t in items:
        p = d.add_paragraph(style="List Number")
        p.add_run(t)
        p.paragraph_format.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY

def _shade(cell, color="D9D9D9"):
    tcPr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:val"), "clear"); shd.set(qn("w:color"), "auto"); shd.set(qn("w:fill"), color)
    tcPr.append(shd)

def table(d, df, widths_cm=None, font_pt=9.5, caption=None):
    if caption:
        cp = d.add_paragraph(); r = cp.add_run(caption); r.italic = True; r.font.size = Pt(10.5)
        cp.paragraph_format.keep_with_next = True
    t = d.add_table(rows=1, cols=len(df.columns))
    t.style = "Table Grid"; t.alignment = WD_TABLE_ALIGNMENT.CENTER
    for i, c in enumerate(df.columns):
        cell = t.rows[0].cells[i]; cell.text = ""
        run = cell.paragraphs[0].add_run(str(c)); run.bold = True; run.font.size = Pt(font_pt)
        _shade(cell)
    for _, row in df.iterrows():
        cells = t.add_row().cells
        for i, v in enumerate(row):
            cells[i].text = ""
            run = cells[i].paragraphs[0].add_run("" if v is None else str(v)); run.font.size = Pt(font_pt)
    for row in t.rows:
        for cell in row.cells:
            for par in cell.paragraphs:
                par.paragraph_format.space_after = Pt(0)
                par.paragraph_format.line_spacing = 1.0
    if widths_cm:
        t.autofit = False
        for i, w in enumerate(widths_cm):
            t.columns[i].width = Cm(w)
        for row in t.rows:
            for i, w in enumerate(widths_cm):
                row.cells[i].width = Cm(w)
    d.add_paragraph().paragraph_format.space_after = Pt(2)
    return t

def figure(d, path, width_cm=15.5, caption=None):
    p = d.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.add_run().add_picture(str(path), width=Cm(width_cm))
    p.paragraph_format.keep_with_next = bool(caption)
    if caption:
        c = d.add_paragraph(); c.alignment = WD_ALIGN_PARAGRAPH.CENTER
        r = c.add_run(caption); r.italic = True; r.font.size = Pt(10.5)

def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"

def fnum(x, k=3):
    return "—" if x is None or (isinstance(x, float) and x != x) else f"{x:.{k}f}"

def fint(x):
    return f"{int(x):,}".replace(",", "\u00a0")


def listing(d, code, font_pt=7.5):
    """Лістинг програми моноширинним шрифтом, по рядку на абзац."""
    for line in code.split("\n"):
        p = d.add_paragraph()
        p.paragraph_format.space_after = Pt(0); p.paragraph_format.line_spacing = 1.0
        r = p.add_run(line if line else " ")
        r.font.name = "Courier New"; r.font.size = Pt(font_pt)
        r._element.rPr.rFonts.set(qn("w:eastAsia"), "Courier New")

## 1. Завантаження бази параметрів квітів Iris

In [ ]:
COLS = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
csv_path = next((p for p in [Path("IrisData_full.csv"), *Path(".").glob("*Iris*.csv")] if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Покладіть IrisData_full.csv у цю папку.")
raw = pd.read_csv(csv_path, header=None, names=COLS + ["species"])
raw["species"] = raw["species"].str.replace("Iris-", "", regex=False)
CLASSES = sorted(raw["species"].unique())
n_dup = int(raw.duplicated().sum())
print(raw.shape, "| класи:", raw["species"].value_counts().to_dict(), "| дублікатів:", n_dup, "| пропусків:", int(raw.isna().sum().sum()))
display(raw.head())

## 2. Перемішування записів

In [ ]:
data = raw.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print("Порядок класів ДО перемішування :", raw["species"].head(8).tolist())
print("Порядок класів ПІСЛЯ перемішування:", data["species"].head(8).tolist())

## 3–4. Розділення на навчальну/тестову вибірки та нормалізація
Нормалізація Min-Max до [0, 1]: z = (x − min) / (max − min). Параметри (min, max) обчислюються **лише на навчальній вибірці**
і застосовуються до тестової — так тестові дані не впливають на підготовку моделі (відсутній «витік» інформації).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

X = data[COLS].to_numpy(); y = data["species"].to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
scaler = MinMaxScaler().fit(X_tr)
Xn_tr, Xn_te = scaler.transform(X_tr), scaler.transform(X_te)
print("train:", Xn_tr.shape, "test:", Xn_te.shape)
print("Частки класів train:", pd.Series(y_tr).value_counts().sort_index().to_dict(), "| test:", pd.Series(y_te).value_counts().sort_index().to_dict())
norm_tab = pd.DataFrame({"ознака": COLS,
                         "min (до)": X_tr.min(0).round(2), "max (до)": X_tr.max(0).round(2),
                         "min (після)": Xn_tr.min(0).round(2), "max (після)": Xn_tr.max(0).round(2)})
display(norm_tab)

## 5. Навчання KNN з різними значеннями K

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

K_RANGE = list(range(1, 31))
rows = []
for k in K_RANGE:
    m = KNeighborsClassifier(n_neighbors=k).fit(Xn_tr, y_tr)
    cv = cross_val_score(KNeighborsClassifier(n_neighbors=k), Xn_tr, y_tr, cv=StratifiedKFold(5, shuffle=True, random_state=SEED)).mean()
    rows.append({"K": k, "train_acc": m.score(Xn_tr, y_tr), "test_acc": m.score(Xn_te, y_te), "cv_acc": cv})
res = pd.DataFrame(rows)
display(res.round(4).head(12))

## 6. Вибір K з найкращими показниками на тестовій вибірці

In [ ]:
best_acc = res["test_acc"].max()
ties = res[res["test_acc"] == best_acc]
K_BEST = int(ties.sort_values(["cv_acc", "K"], ascending=[False, True]).iloc[0]["K"])   # при рівності — кращий за крос-валідацією
print(f"Максимальна точність на тесті: {best_acc:.4f}; K з такою точністю: {ties['K'].tolist()}; обрано K = {K_BEST}")
final = KNeighborsClassifier(n_neighbors=K_BEST).fit(Xn_tr, y_tr)
pred = final.predict(Xn_te)
from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_te, pred, labels=CLASSES)
rep = pd.DataFrame(classification_report(y_te, pred, output_dict=True, zero_division=0)).T.loc[CLASSES + ["macro avg"]]
display(cm); display(rep.round(3))

## 7. Стійкість вибору K: 200 випадкових розбиттів
Тест із 45 квітів дуже малий (один об’єкт = 2,2% точності), тому перевіряємо, як K поводиться в середньому.

In [ ]:
from sklearn.pipeline import make_pipeline
splits = []
for s in range(200):
    a, b, c, d_ = train_test_split(X, y, test_size=0.30, stratify=y, random_state=1000 + s)
    for k in K_RANGE:
        mp = make_pipeline(MinMaxScaler(), KNeighborsClassifier(n_neighbors=k)).fit(a, c)
        splits.append((k, mp.score(b, d_)))
rep_df = pd.DataFrame(splits, columns=["K", "acc"]).groupby("K")["acc"].agg(["mean", "std"]).reset_index()
K_STABLE = int(rep_df.loc[rep_df["mean"].idxmax(), "K"])
acc_stable = float(rep_df["mean"].max())
good_range = rep_df[rep_df["mean"] >= acc_stable - 0.005]["K"].tolist()
print(f"Найкращий у середньому K = {K_STABLE} (acc = {acc_stable:.4f}); K у межах 0,5 п.п.: {good_range}")

## 8. Власна реалізація k-NN (NumPy) для перевірки

In [ ]:
def knn_predict(Xtr, ytr, Xq, k):
    """Евклідова відстань до всіх навчальних точок -> k найближчих -> голосування більшістю."""
    d = np.sqrt(((Xq[:, None, :] - Xtr[None, :, :]) ** 2).sum(axis=2))
    idx = np.argsort(d, axis=1, kind="stable")[:, :k]
    out = []
    for row in idx:
        labels, counts = np.unique(ytr[row], return_counts=True)
        out.append(labels[np.argmax(counts)])
    return np.array(out)

own_pred = knn_predict(Xn_tr, y_tr, Xn_te, K_BEST)
own_acc = float((own_pred == y_te).mean())
agree = float((own_pred == pred).mean())
print(f"власна реалізація: acc = {own_acc:.4f}; збіг із sklearn: {agree:.1%}")

## 9. Додаткове завдання (hometask): синтетичний набір make_blobs

In [ ]:
from sklearn.datasets import make_blobs
Xs, ys = make_blobs(n_samples=300, n_features=2, centers=8, cluster_std=1.3, random_state=4)
ys = ys % 2
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(Xs, ys, test_size=0.25, stratify=ys, random_state=SEED)
sc2 = MinMaxScaler().fit(Xs_tr); Xs_trn, Xs_ten = sc2.transform(Xs_tr), sc2.transform(Xs_te)
srows = []
for k in range(1, 41):
    mm = KNeighborsClassifier(n_neighbors=k).fit(Xs_trn, ys_tr)
    srows.append({"K": k, "train_acc": mm.score(Xs_trn, ys_tr), "test_acc": mm.score(Xs_ten, ys_te)})
sres = pd.DataFrame(srows)
s_best_acc = sres["test_acc"].max()
S_K_BEST = int(sres[sres["test_acc"] == s_best_acc]["K"].min())
print(f"Синтетичний набір: найкращий K = {S_K_BEST}, test acc = {s_best_acc:.4f}; для K=1: train {sres.loc[0,'train_acc']:.3f} / test {sres.loc[0,'test_acc']:.3f}")

## 10. Графіки

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(res["K"], res["train_acc"], marker="o", ms=3, label="навчальна")
ax[0].plot(res["K"], res["test_acc"], marker="s", ms=3, label="тестова")
ax[0].plot(res["K"], res["cv_acc"], marker="^", ms=3, ls="--", label="5-fold CV (train)")
ax[0].axvline(K_BEST, color="gray", ls=":"); ax[0].set_xlabel("K"); ax[0].set_ylabel("Accuracy"); ax[0].set_title("Iris: точність залежно від K"); ax[0].legend(fontsize=8)
ax[1].errorbar(rep_df["K"], rep_df["mean"], yerr=rep_df["std"], fmt="o-", ms=3, capsize=2, color="seagreen")
ax[1].axvline(K_STABLE, color="gray", ls=":"); ax[1].set_xlabel("K"); ax[1].set_ylabel("Середня accuracy ± std"); ax[1].set_title("200 випадкових розбиттів")
plt.tight_layout(); plt.savefig(FIG / "k_curve.png", dpi=170); plt.show()

def boundary(ax, Xa, ya, k, xlab, ylab, title, classes):
    from matplotlib.colors import ListedColormap
    cm_light = ListedColormap(["#cfe0f5", "#fde2c8", "#d3efd3"][:len(classes)])
    cm_bold = ListedColormap(["#1f5fa8", "#e07b00", "#2a8a2a"][:len(classes)])
    mdl = KNeighborsClassifier(n_neighbors=k).fit(Xa, ya)
    xx, yy = np.meshgrid(np.linspace(Xa[:, 0].min() - .05, Xa[:, 0].max() + .05, 300), np.linspace(Xa[:, 1].min() - .05, Xa[:, 1].max() + .05, 300))
    zz = mdl.predict(np.c_[xx.ravel(), yy.ravel()])
    zz = pd.Series(zz).map({c: i for i, c in enumerate(classes)}).to_numpy().reshape(xx.shape)
    ax.pcolormesh(xx, yy, zz, cmap=cm_light, shading="auto")
    yi = pd.Series(ya).map({c: i for i, c in enumerate(classes)}).to_numpy()
    ax.scatter(Xa[:, 0], Xa[:, 1], c=yi, cmap=cm_bold, s=16, edgecolor="k", linewidth=0.3)
    ax.set_xlabel(xlab); ax.set_ylabel(ylab); ax.set_title(title)

fig, ax = plt.subplots(1, 3, figsize=(13.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[0])
ax[0].set_xlabel("Передбачено"); ax[0].set_ylabel("Фактично"); ax[0].set_title(f"Матриця помилок, K = {K_BEST}")
boundary(ax[1], Xn_tr[:, 2:4], y_tr, K_BEST, "petal_length (норм.)", "petal_width (норм.)", f"Iris: межі рішень, K = {K_BEST}", CLASSES)
boundary(ax[2], Xs_trn, ys_tr, S_K_BEST, "x₁ (норм.)", "x₂ (норм.)", f"make_blobs: межі рішень, K = {S_K_BEST}", [0, 1])
plt.tight_layout(); plt.savefig(FIG / "boundary.png", dpi=170); plt.show()

## 11. Автоматичне формування Word-звіту

In [ ]:
def pc(x): return f"{x * 100:.1f}".replace(".", ",") + "%"

PROGRAM_LISTING = '''import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier

# 1. Завантаження бази Iris (4 ознаки + клас)
cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
raw = pd.read_csv("IrisData_full.csv", header=None, names=cols + ["species"])

# 2. Перемішування записів
data = raw.sample(frac=1.0, random_state=42).reset_index(drop=True)
X, y = data[cols].to_numpy(), data["species"].to_numpy()

# 3. Розділення на навчальну (70%) і тестову (30%) вибірки зі збереженням часток класів
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)

# 4. Нормалізація Min-Max: z = (x - min) / (max - min); параметри — лише з навчальної вибірки
scaler = MinMaxScaler().fit(X_tr)
Xn_tr, Xn_te = scaler.transform(X_tr), scaler.transform(X_te)

# 5. Навчання k-NN-класифікатора для K = 1..30 та оцінка на тестовій вибірці
scores = {}
for k in range(1, 31):
    model = KNeighborsClassifier(n_neighbors=k).fit(Xn_tr, y_tr)
    scores[k] = model.score(Xn_te, y_te)

# 6. Вибір K з найкращою якістю класифікації на тестовій вибірці
k_best = max(scores, key=scores.get)
print("The best k =", k_best, ", score =", scores[k_best])

# 7. Власна реалізація k-NN (евклідова відстань + голосування більшістю)
def knn_predict(Xtr, ytr, Xq, k):
    d = np.sqrt(((Xq[:, None, :] - Xtr[None, :, :]) ** 2).sum(axis=2))   # відстані до всіх точок
    idx = np.argsort(d, axis=1, kind="stable")[:, :k]                     # k найближчих сусідів
    return np.array([np.unique(ytr[r], return_counts=True)[0][np.argmax(np.unique(ytr[r], return_counts=True)[1])] for r in idx])'''

# перевірка, що лістинг справді відтворює результат основного аналізу
_ns = {}
exec(PROGRAM_LISTING, _ns)
assert _ns["k_best"] == int(res.sort_values(["test_acc", "K"], ascending=[False, True]).iloc[0]["K"]), "лістинг не збігається з аналізом"
assert abs(_ns["scores"][_ns["k_best"]] - best_acc) < 1e-12

n_ok = int(round(best_acc * len(y_te))); n_te = len(y_te)
miss = [(a, b) for a, b in zip(y_te, pred) if a != b]
miss_txt = ", ".join(f"{a} → {b}" for a, b in miss) if miss else "помилок немає"
k_small, k_big = res.iloc[0], res.iloc[-1]
show_k = sorted(set([1, 2, 3, 5, 7, 9, 10, 11, 12, 15, 20, 25, 30]))
k_tab = res[res["K"].isin(show_k)].copy()
k_tab = pd.DataFrame({"K": k_tab["K"].astype(int), "Навчальна": k_tab["train_acc"].map(pc), "Тестова": k_tab["test_acc"].map(pc), "CV (5-fold)": k_tab["cv_acc"].map(pc)})
k_tab["Обрано"] = ["◄ K = %d" % K_BEST if int(k) == K_BEST else "" for k in k_tab["K"]]

doc = new_doc(3, "ДОСЛІДЖЕННЯ ПОБУДОВИ КЛАСИФІКАТОРА K‑НАЙБЛИЖЧИХ СУСІДІВ (K‑NN)", "", "", STUDENT_NAME, GROUP_NAME,
              discipline="Методи та системи штучного інтелекту", info="Набір даних: Iris (150 квіток, 3 класи)")

P(doc, "Дослідження побудови класифікатора k-найближчих сусідів (k-nn).", bold_prefix="Тема: ")
P(doc, "набути навичок щодо можливостей аналізу даних з використанням класифікатора методом k-найближчих сусідів (k-nn).", bold_prefix="Мета роботи: ")

H(doc, "1. Теоретичні відомості")
P(doc, "Метод k-найближчих сусідів — найпростіший метричний класифікатор, що ґрунтується на оцінюванні подібності об’єктів. Він не будує явної моделі: "
       "для нового об’єкта обчислюється відстань до всіх об’єктів навчальної вибірки, відбираються k найближчих і клас визначається простим голосуванням більшістю. "
       "Як відстань зазвичай використовують евклідову: d(a, b) = √(Σᵢ (aᵢ − bᵢ)²), де i — номер ознаки.")
P(doc, "Єдиний параметр методу — K. Малі K дають складну межу між класами та чутливість до шуму (перенавчання), великі — надмірне згладжування (недонавчання); "
       "при K, що дорівнює розміру навчальної вибірки, прогноз завжди збігається з найчастішим класом. Оскільки метод працює з відстанями, ознаки мають бути в порівнянних масштабах, "
       "тому застосовують нормалізацію, зокрема Min-Max: z = (x − min) / (max − min).")

H(doc, "2. Хід роботи")
P(doc, f"Використано файл IrisData_full.csv: {len(raw)} квітів, 4 ознаки (довжина й ширина чашолистка та пелюстки, см), три класи по {raw['species'].value_counts().iloc[0]} об’єктів "
       f"({', '.join(CLASSES)}). Пропусків немає; {n_dup} повні дублікати залишено, бо вони є природними повторами вимірювань у цьому відомому наборі. "
       "Для програмної реалізації обрано мову Python (scikit-learn, NumPy).", bold_prefix="2.1. Завантаження бази. ")
table(doc, raw.head(5).rename(columns={"sepal_length": "sepal length", "sepal_width": "sepal width", "petal_length": "petal length", "petal_width": "petal width"}),
      widths_cm=[3.0, 3.0, 3.0, 3.0, 3.5], font_pt=9.5, caption="Таблиця 1. Перші записи бази Iris (до перемішування)")
P(doc, f"Записи впорядковані за класами (спочатку всі setosa), тому перед поділом їх перемішано (генератор із фіксованим seed = {SEED} для відтворюваності). "
       f"Після перемішування класи чергуються: {', '.join(data['species'].head(4))} …", bold_prefix="2.2. Перемішування. ")
P(doc, f"Дані розділено на навчальну ({len(y_tr)} квіток, 70%) і тестову ({len(y_te)} квіток, 30%) вибірки зі збереженням часток класів (по {int((y_tr == CLASSES[0]).sum())} і {int((y_te == CLASSES[0]).sum())} об’єктів кожного класу). "
       "Ознаки нормалізовано методом Min-Max до [0, 1]; min і max обчислено лише за навчальною вибіркою й застосовано до тестової, щоб тестові дані не впливали на підготовку моделі. "
       "Це суттєво, бо ознаки мають різні діапазони (наприклад, petal length — 1,1…6,9 см, petal width — 0,1…2,5 см).", bold_prefix="2.3. Розділення на вибірки та нормалізація. ")
table(doc, norm_tab, widths_cm=[3.4, 2.6, 2.6, 2.8, 2.8], font_pt=9.5, caption="Таблиця 2. Діапазони ознак навчальної вибірки до та після нормалізації")
P(doc, f"Класифікатор навчено для K = 1…{K_RANGE[-1]} (евклідова відстань, невиважене голосування). Для кожного K обчислено точність на навчальній і тестовій вибірках, "
       "а також середню точність 5-кратної перехресної перевірки на навчальній вибірці (Таблиця 3, Рис. 1).", bold_prefix="2.4. Навчання для різних K. ")
table(doc, k_tab, widths_cm=[1.6, 3.2, 3.2, 3.4, 3.0], font_pt=9.5, caption="Таблиця 3. Точність класифікації залежно від K (вибрані значення)")
figure(doc, FIG / "k_curve.png", 16, "Рис. 1. Залежність точності від K (ліворуч) та стійкість вибору K на 200 випадкових розбиттях (праворуч)")
tie_txt = ("єдине значення з максимальною точністю" if len(ties) == 1 else f"серед K = {', '.join(map(str, ties['K']))} з однаковою точністю обрано кращий за крос-валідацією")
P(doc, f"Найвищу точність на тестовій вибірці — {pc(best_acc)} ({n_ok} з {n_te} квіток) — дає K = {K_BEST} ({tie_txt}). "
       f"На матриці помилок (Рис. 2) {'лише одна помилка' if len(miss) == 1 else str(len(miss)) + ' помилок'}: {miss_txt}; клас setosa розпізнається безпомилково, "
       "адже лінійно відокремлений від двох інших. Метрики за класами: " +
       "; ".join(f"{c}: precision {rep.loc[c, 'precision']:.2f}, recall {rep.loc[c, 'recall']:.2f}" for c in CLASSES) + ".", bold_prefix="2.5. Вибір K. ")
figure(doc, FIG / "boundary.png", 16.5, f"Рис. 2. Матриця помилок для K = {K_BEST}; межі рішень k-NN для Iris (у площині ознак пелюстки) та для синтетичного набору")
P(doc, f"Тестова вибірка містить лише {n_te} об’єктів, тому одна квітка змінює точність на {100 / n_te:.1f}".replace(".", ",") + " п.п., і різниця між сусідніми K часто є статистичним шумом. "
       f"Для перевірки стійкості вибір повторено на 200 випадкових розбиттях (нормалізація в кожному — лише за навчальною частиною). Найвища середня точність — {pc(acc_stable)} при K = {K_STABLE}; "
       f"близьку якість (у межах 0,5 п.п.) мають усі K = {good_range[0]}…{good_range[-1]}. Отже, K = {K_BEST} — коректний, але не єдиний хороший вибір, а надійною зоною є помірні K.", bold_prefix="2.6. Стійкість вибору K. ")
P(doc, f"Щоб переконатися в розумінні алгоритму, його реалізовано вручну на NumPy (відстані до всіх точок, k найближчих, голосування). "
       f"Для K = {K_BEST} власна реалізація дала точність {pc(own_acc)} і збіглася зі scikit-learn у {pc(agree)} прогнозів.", bold_prefix="2.7. Власна реалізація. ")
P(doc, f"Для синтетичного набору make_blobs (300 точок, 2 ознаки, 8 центрів, клас = номер центру mod 2; класи нелінійно роздільні) підібрано K аналогічно. "
       f"Найкращий K = {S_K_BEST} дає точність {pc(s_best_acc)} на тесті; при K = 1 точність на навчальній вибірці {pc(sres.loc[0, 'train_acc'])}, а на тестовій {pc(sres.loc[0, 'test_acc'])} — "
       "межа рішень стає рваною й «підганяється» під окремі точки (Рис. 2, праворуч).", bold_prefix="2.8. Додаткове завдання (hometask). ")

H(doc, "3. Висновки")
numbered(doc, [
    f"Побудовано k-NN-класифікатор для Iris за всіма кроками завдання: завантаження, перемішування, нормалізація, розділення на вибірки, навчання для K = 1…{K_RANGE[-1]} та вибір K.",
    f"На тестовій вибірці найкращий результат — {pc(best_acc)} при K = {K_BEST}; єдина помилка стосується пари versicolor/virginica, що частково перекриваються у просторі ознак, а setosa відокремлюється ідеально.",
    f"Поведінка при крайніх K відповідає теорії: K = 1 дає {pc(k_small['train_acc'])} на навчальній вибірці (запам’ятовування), K = {K_RANGE[-1]} — {pc(k_big['test_acc'])} на тесті (згладжування). Найкраща якість — у середній зоні.",
    f"Через малий тест вибір K потребує обережності: за 200 розбиттями найкращий у середньому K = {K_STABLE} ({pc(acc_stable)}), а хороші значення лежать у діапазоні {good_range[0]}…{good_range[-1]}.",
    "Нормалізація необхідна, бо метод спирається на відстані; її параметри обчислюються лише на навчальних даних. Метод простий і не потребує навчання, але зберігає всю вибірку та обчислює відстані до кожної точки, що ускладнює роботу на великих даних.",
])

H(doc, "Додаток. Лістинг програми")
P(doc, "Програма на Python (повна версія з графіками та формуванням звіту — у файлі MSHI_lab03.ipynb).", italic=True)
listing(doc, PROGRAM_LISTING)

report_file = OUT / "MSHI_lab03_report.docx"
doc.save(report_file)
print("Звіт збережено:", report_file.resolve())